### PREPROCESSING, EVALUATION, PIPELINES & MODEL SAVING

In [1]:
# Titanic Mini Project


import pandas as pd
import seaborn as sns
import joblib
from sklearn.model_selection import ( StratifiedKFold, cross_val_score, GridSearchCV)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Load Dataset

df = sns.load_dataset('titanic')

# Drop unwanted columns

df = df.drop(columns=['deck','embark_town','alive','who','adult_male','class' ])

# Features and Target

X = df.drop('survived', axis=1)
y = df['survived']

# Use only specified features
X = X[['age', 'fare', 'parch', 'sibsp','sex', 'embarked', 'pclass']]

# Column Groups

numerical_features = ['age', 'fare', 'parch', 'sibsp']
categorical_features = ['sex', 'embarked']
ordinal_features = ['pclass']

# Preprocessing

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

ordinal_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features),
    ('ord', ordinal_transformer, ordinal_features)
])

# Pipeline

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier( random_state=42 )])

# ==========================================
# 5-Fold Stratified Cross Validation
# ==========================================

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

scores = cross_val_score(pipeline,X,y,cv=cv,scoring='accuracy')

print("Cross Validation Accuracy:")
print(f"Mean Accuracy = {scores.mean():.4f}")
print(f"Std Accuracy  = {scores.std():.4f}")
print(f"Mean ± Std    = {scores.mean():.4f} ± {scores.std():.4f}")

# Expected: around 0.80 - 0.83

# Grid Search

param_grid = {'model__n_estimators': [50, 100, 200],'model__max_depth': [5, 10, None]}

grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)

grid_search.fit(X, y)

print("\nBest Parameters:")
print(grid_search.best_params_)

print("\nBest CV Accuracy:")
print(grid_search.best_score_)

# Best Model


best_pipeline = grid_search.best_estimator_

# Save Model

joblib.dump(best_pipeline, 'titanic_best_model.pkl')

print("\nModel saved successfully.")

# Load Model

loaded_model = joblib.load('titanic_best_model.pkl')

# ==========================================
# Prediction
# 30-year-old male
# pclass = 3
# embarked = S
# ==========================================

new_passenger = pd.DataFrame({
    'age': [30],
    'fare': [7.25],      # typical class-3 fare
    'parch': [0],
    'sibsp': [0],
    'sex': ['male'],
    'embarked': ['S'],
    'pclass': [3]
})

prediction = loaded_model.predict(new_passenger)[0]
probability = loaded_model.predict_proba(new_passenger)[0]

print("\nPrediction:")
print("Survived" if prediction == 1 else "Did Not Survive")

print("\nProbabilities:")
print(f"Not Survive: {probability[0]:.4f}")
print(f"Survive    : {probability[1]:.4f}")

Cross Validation Accuracy:
Mean Accuracy = 0.8125
Std Accuracy  = 0.0192
Mean ± Std    = 0.8125 ± 0.0192

Best Parameters:
{'model__max_depth': 10, 'model__n_estimators': 200}

Best CV Accuracy:
0.8428598330299415

Model saved successfully.

Prediction:
Did Not Survive

Probabilities:
Not Survive: 0.9295
Survive    : 0.0705
